In [18]:
import modin.pandas as pd
import numpy as np
import os
import scipy.stats as stats

In [52]:
jobs_data_dir = "Job Data\\"
dataframes = []
# Read all files in the directory
for file_name in os.listdir(jobs_data_dir):
    if '2024' in file_name: 
        print(file_name)
        file_path = os.path.join(jobs_data_dir, file_name)
        tmp_df = pd.read_csv(file_path, usecols = ['jobPostingId', 'title', 'description', 'tokens', 'gender_category'])
        tmp_df = tmp_df.dropna(how='any')
        dataframes.append(tmp_df)
job_postings_df = pd.concat(dataframes)
print(len(job_postings_df))
job_postings_df.head()

annotated_data_2024_batch_2_removed_words_tokenized.csv
annotated_data_2024_removed_words_tokenized.csv
77975


,jobPostingId,title,description,gender_category,tokens
0,3823301365,Python Full Stack Developer/Lead/Architect,"Skills- Python, Django, React/AngularExperienc...",fem,"['python', 'full', 'stack', 'python', 'django'..."
1,3823301077,Senior Data Engineer - w2 only,Expertise:8+ years of relevant industry exper...,neutral,"['senior', 'data', 'engineer', 'year', 'releva..."
2,3823301053,Retail Front End Supervisor,LOCATION 1130 Newport Avenue Attleboro MA US 0...,masc,"['retail', 'front', 'end', 'supervisor', 'loca..."
3,3823301005,"Front-end Software Engineer, LATAM",Build the platform behind the most extensive s...,masc,"['software', 'engineer', 'latam', 'build', 'pl..."
4,3823300903,Senior Python Developer with DevOps Experience,Tittle: Python Developer with DevOps Experienc...,masc,"['senior', 'python', 'developer', 'devops', 'e..."


In [55]:
recommendations_df = pd.read_csv('Recommendation Data/tf_idf_160000_2.csv')
print(len(recommendations_df))
recommendations_df.head()

160000


,Unnamed: 0,cv_index,CV,applicant_gender,jobPostingId,title,description,gender_category,rank
0,0,0,```plaintext\n**Summary** \nDynamic and detai...,female,3822726679,Dotnet Developer,**Job Title: Junior/Mid Software Developer (as...,fem,1
1,1,0,```plaintext\n**Summary** \nDynamic and detai...,female,3850910575,ASP.NET Developer,The ASP.NET Developer is responsible for the d...,fem,2
2,2,0,```plaintext\n**Summary** \nDynamic and detai...,female,3879707553,Node.js Developer,We are seeking an experienced Senior Node.js D...,fem,3
3,3,0,```plaintext\n**Summary** \nDynamic and detai...,female,3966594528,Full Stack Engineer-NET,How you should be?\nWe are looking for a talen...,fem,4
4,4,0,```plaintext\n**Summary** \nDynamic and detai...,female,4090470487,.NET Software Engineer with React,"Skills:\nC#, ASP.NET, React, JavaScript, SQL, ...",neutral,5


In [56]:
print(len(recommendations_df[recommendations_df['gender_category']=='fem']), len(recommendations_df[recommendations_df['gender_category']=='masc']), len(recommendations_df[recommendations_df['gender_category']=='neutral']))

135186 22396 2418


## Disparity-based exposure fairness

In [61]:
# Calculate exposure per item (count of users the item was recommended to)
exposure_df = recommendations_df.groupby(['jobPostingId', 'gender_category'])['cv_index'].count().reset_index()
exposure_df.rename(columns={'cv_index': 'exposure'}, inplace=True)
exposure_df.head()

,jobPostingId,gender_category,exposure
0,3822726679,fem,927
1,3822727195,neutral,49
2,3822732751,fem,1
3,3822735151,fem,2
4,3822737443,masc,16


In [62]:
# Split by gender category
group_1 = exposure_df[exposure_df['gender_category'] == 'masc']  # e.g., ''
group_2 = exposure_df[exposure_df['gender_category'] == 'fem']  # e.g., 'fem'

In [63]:
# Calculate total exposure for each group
exposure_g1 = group_1['exposure'].sum()
exposure_g2 = group_2['exposure'].sum()

# Compute Extract-K-based fairness (alpha)
alpha = exposure_g1 / exposure_g2
alpha

0.16566804254878464

In [64]:
# Compute Disparity-based Exposure Fairness
exposure_disparity = abs(exposure_g1/len(group_1) - alpha * exposure_g2/len(group_2))
exposure_disparity

71.24611186903138

## Jain's Fairness Index

In [184]:
def jains_fairness_index_from_groups(df, group_col='gender_category', exposure_col='exposure'):
    """
    Compute Jain's Fairness Index over two item groups (e.g., 'masculine' and 'feminine').

    df: DataFrame with at least two columns:
        - group_col: e.g., 'gender_category' with values 'masculine' and 'feminine'
        - exposure_col: e.g., 'exposure' representing exposure counts per job

    Returns:
        Jain's Fairness Index (float between 0 and 1)
    """
    # Aggregate total exposure per group
    group_exposures = df.groupby(group_col)[exposure_col].sum()

    # Extract exposures safely
    x1 = group_exposures.get('masc', 0)
    x2 = group_exposures.get('fem', 0)

    numerator = (x1 + x2) ** 2
    denominator = 2 * (x1**2 + x2**2)

    return numerator / denominator if denominator != 0 else 0


In [185]:
# Assuming your DataFrame is named job_exposure_df
jfi_score = jains_fairness_index_from_groups(job_exposure)
print("Jain's Fairness Index:", round(jfi_score, 4))

Jain's Fairness Index: 0.6612


## User-side

In [22]:
# Assuming recommendations_df contains:
# 'cv_index', 'job_id', 'job_gender_code', 'applicant_gender'
# Where 'applicant_gender' ∈ {'male', 'female'}
# and 'job_gender_code' ∈ {'masculine', 'feminine'}

def recommendation_bias_metrics(recommendations_df):
    results = {}

    for group in ['male', 'female']:
        group_df = recommendations_df[recommendations_df['applicant_gender'] == group]
        total_recs = len(group_df)
        masc_count = len(group_df[group_df['gender_category'] == 'masc'])
        fem_count = len(group_df[group_df['gender_category'] == 'fem'])

        results[f'{group}_masc_pct'] = masc_count / total_recs if total_recs > 0 else 0
        results[f'{group}_fem_pct'] = fem_count / total_recs if total_recs > 0 else 0

    # Differences in recommendation propensities
    results['masc_bias_diff'] = results['male_masc_pct'] - results['female_masc_pct']
    results['fem_bias_diff'] = results['female_fem_pct'] - results['male_fem_pct']

    return results


In [23]:
results = recommendation_bias_metrics(recommendations_df)

In [188]:
results

{'male_masc_pct': 0.159,
 'male_fem_pct': 0.82665,
 'female_masc_pct': 0.12095,
 'female_fem_pct': 0.863175,
 'masc_bias_diff': 0.03805,
 'fem_bias_diff': 0.03652500000000003}

In [189]:
# male_masc_pct = proportion of jobs recommended to men that are masculine

# female_masc_pct = same for women

# masc_bias_diff:

# Positive → masculine jobs shown more to men

# Negative → masculine jobs shown more to women

# Same logic applies to fem_bias_diff.

## Correlation

In [57]:
# Create a contingency table
contingency_table = pd.crosstab(recommendations_df['applicant_gender'], recommendations_df['gender_category'])
contingency_table.head()

col_0,fem,masc,neutral
row_0,,,
female,69054,9676,1270
male,66132,12720,1148


In [58]:
def cramers_v(confusion_matrix):
    """Calculate Cramér's V statistic from a confusion matrix."""
    chi2, p, dof, expected = stats.chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    v = np.sqrt(phi2 / min(k - 1, r - 1))
    return v, p

In [59]:
# Stratification logic
rank_groups = {
    'rank_1': recommendations_df[recommendations_df['rank'] == 1],
    'rank_1_to_5': recommendations_df[recommendations_df['rank'] <= 5],
    'rank_1_to_10': recommendations_df[recommendations_df['rank'] <= 10]
}

In [60]:
# Analyze each group
for name, group in rank_groups.items():
    contingency = pd.crosstab(group['applicant_gender'], group['gender_category'])
    v, p = cramers_v(contingency)
    print(f"--- {name} ---")
    print(f"Cramér's V: {v:.3f}")
    print(f"p-value: {p:.3e}")
    print()

--- rank_1 ---
Cramér's V: 0.084
p-value: 3.714e-25

--- rank_1_to_5 ---
Cramér's V: 0.060
p-value: 1.468e-62

--- rank_1_to_10 ---
Cramér's V: 0.055
p-value: 1.282e-105

